In [1]:
import pandas as pd
from shared_modeling import load_or_create_master_split_ids, run_model_experiment


In [2]:
columns = ['V2AF13', 'V2AF15', 'PublicID'] # father's age and years of education
df_paternal = pd.read_csv('../Data/V2A.csv', usecols=columns)
df_outcomes = pd.read_csv('../Data/Modified/Outcome.csv', usecols=['PublicID', 'MH_outcome'])

# Create the master split once and persist it for reuse in other notebooks.
split_path = 'master_split_ids.csv'
train_ids, test_ids = load_or_create_master_split_ids(df_outcomes, split_path)
df_outcomes

,PublicID,MH_outcome
0,00004O,1
1,00007I,1
2,00008G,0
3,00015J,0
4,00016H,1
...,...,...
7736,17349I,1
7737,17350A,1
7738,17351V,0
7739,17352T,1


In [3]:
df = pd.merge(df_paternal, df_outcomes, on='PublicID', how='inner')
df.shape

(7608, 4)

In [4]:
# paternal age feature
df['V2AF13'].value_counts()

V2AF13
30    535
31    493
32    492
28    464
29    459
33    414
27    412
25    381
26    376
24    368
34    351
23    330
35    284
22    257
36    230
21    221
37    199
20    170
19    149
38    139
39     96
40     95
18     94
41     81
42     57
17     46
43     37
45     33
D      32
44     30
46     26
47     22
16     17
49     15
48     13
50     11
52      7
15      5
59      2
55      2
70      2
13      2
54      1
58      1
56      1
64      1
57      1
69      1
Name: count, dtype: int64

In [5]:
df = df[df != 'R']
df = df[df != 'D']
df['V2AF13'] = pd.to_numeric(df['V2AF13'], errors='coerce')
df['V2AF15'] = pd.to_numeric(df['V2AF15'], errors='coerce')
df

,PublicID,V2AF13,V2AF15,MH_outcome
0,00004O,26.0,5.0,1
1,00007I,39.0,3.0,1
2,00008G,32.0,3.0,0
3,00015J,33.0,6.0,0
4,00016H,29.0,3.0,1
...,...,...,...,...
7603,17349I,25.0,5.0,1
7604,17350A,38.0,3.0,1
7605,17351V,32.0,6.0,0
7606,17352T,31.0,6.0,1


In [6]:
X = df.drop(['MH_outcome', 'PublicID'], axis=1)
y = df['MH_outcome']

train_df = df[df['PublicID'].isin(train_ids)].copy()
test_df = df[df['PublicID'].isin(test_ids)].copy()

X_train = train_df.drop(['MH_outcome', 'PublicID'], axis=1)
X_test = test_df.drop(['MH_outcome', 'PublicID'], axis=1)
y_train = train_df['MH_outcome']
y_test = test_df['MH_outcome']

y.value_counts()

MH_outcome
0    4517
1    3091
Name: count, dtype: int64

In [7]:
X

,V2AF13,V2AF15
0,26.0,5.0
1,39.0,3.0
2,32.0,3.0
3,33.0,6.0
4,29.0,3.0
...,...,...
7603,25.0,5.0
7604,38.0,3.0
7605,32.0,6.0
7606,31.0,6.0


In [8]:
y.value_counts()

MH_outcome
0    4517
1    3091
Name: count, dtype: int64

In [9]:
# Run the LR pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'lr',
    ['V2AF13', 'V2AF15']
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters found: {'classifier__C': 0.001, 'classifier__l1_ratio': 0.25}
Best Macro F1 Score: 0.5572
Model Coefficients:
num__V2AF13: 0.0
num__V2AF15: -0.1002481113729886
Evaluation Metrics for LR with shared preprocessing and macro F1 grid search:
Accuracy: 0.5807
Precision: 0.5661
Recall: 0.5663
F1-score: 0.5662
ROC AUC: 0.5781


In [10]:
# Run the RF pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'rf',
    ['V2AF13', 'V2AF15']
)

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters found: {'classifier__max_depth': 20, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 700}
Best Macro F1 Score: 0.5412
Feature Importances:
num__V2AF13: 0.6473107818119003
num__V2AF15: 0.3526892181880996
Evaluation Metrics for RF with shared preprocessing and macro F1 grid search:
Accuracy: 0.5538
Precision: 0.5531
Recall: 0.5550
F1-score: 0.5496
ROC AUC: 0.5716


In [11]:
# Run the XGBoost pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'xgb',
    ['V2AF13', 'V2AF15']
)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found: {'classifier__colsample_bytree': 1.0, 'classifier__learning_rate': 0.01, 'classifier__max_depth': 4, 'classifier__n_estimators': 100, 'classifier__subsample': 0.7}
Best Macro F1 Score: 0.5560
Feature Importances:
num__V2AF13: 0.16138671338558197
num__V2AF15: 0.8386132717132568
Evaluation Metrics for XGB with shared preprocessing and macro F1 grid search:
Accuracy: 0.5748
Precision: 0.5717
Recall: 0.5742
F1-score: 0.5696
ROC AUC: 0.5868


In [12]:
# Run the SVM pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'svm',
    ['V2AF13', 'V2AF15']
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters found: {'classifier__estimator__C': 0.1, 'classifier__estimator__gamma': 0.1, 'classifier__estimator__kernel': 'rbf'}
Best Macro F1 Score: 0.5573
Evaluation Metrics for SVM with shared preprocessing and macro F1 grid search:
Accuracy: 0.5715
Precision: 0.5617
Recall: 0.5631
F1-score: 0.5616
ROC AUC: 0.5745
